In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

In [2]:
spark = SparkSession\
    .builder\
    .master("local[*]")\
    .appName("Files")\
    .config("spark.log.level", "WARN")\
    .config("spark.jars", "/opt/avro/spark-avro_2.12-3.5.8.jar")\
    .config("spark.sql.debug.maxToStringFields", 1024)\
    .getOrCreate()

26/06/28 16:04:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Setting Spark log level to "WARN".
26/06/28 16:04:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
print("spark.version == ", spark.version)

spark.version ==  3.5.8


## CSV

### Пример 1

In [4]:
!cat data/people.csv

name;age;job
Jorge;30;Developer
Bob;32;Developer


In [5]:
path = "data/people.csv"
df1 = spark.read.csv(path)
df1.show()

+------------------+
|               _c0|
+------------------+
|      name;age;job|
|Jorge;30;Developer|
|  Bob;32;Developer|
+------------------+



In [6]:
df2 = spark.read.option("delimiter", ";").csv(path)
df2.show()

+-----+---+---------+
|  _c0|_c1|      _c2|
+-----+---+---------+
| name|age|      job|
|Jorge| 30|Developer|
|  Bob| 32|Developer|
+-----+---+---------+



In [7]:
df3 = spark.read.option("delimiter", ";").option("header", "true").csv(path)
df3.show()

+-----+---+---------+
| name|age|      job|
+-----+---+---------+
|Jorge| 30|Developer|
|  Bob| 32|Developer|
+-----+---+---------+



In [8]:
df4 = spark.read.options(delimiter=";", header=True).csv(path)
df4.show()

+-----+---+---------+
| name|age|      job|
+-----+---+---------+
|Jorge| 30|Developer|
|  Bob| 32|Developer|
+-----+---+---------+



### Пример 2

In [9]:
!head data/books.csv

Name,Author,User Rating,Reviews,Price,Year,Genre
10-Day Green Smoothie Cleanse,JJ Smith,4.7,17350,8,2016,Non Fiction
11/22/63: A Novel,Stephen King,4.6,2052,22,2011,Fiction
12 Rules for Life: An Antidote to Chaos,Jordan B. Peterson,4.7,18979,15,2018,Non Fiction
1984 (Signet Classics),George Orwell,4.7,21424,6,2017,Fiction
"5,000 Awesome Facts (About Everything!) (National Geographic Kids)",National Geographic Kids,4.8,7665,12,2019,Non Fiction
A Dance with Dragons (A Song of Ice and Fire),George R. R. Martin,4.4,12643,11,2011,Fiction
A Game of Thrones / A Clash of Kings / A Storm of Swords / A Feast of Crows / A Dance with Dragons,George R. R. Martin,4.7,19735,30,2014,Fiction
A Gentleman in Moscow: A Novel,Amor Towles,4.7,19699,15,2017,Fiction
"A Higher Loyalty: Truth, Lies, and Leadership",James Comey,4.7,5983,3,2018,Non Fiction


In [10]:
booksPath = "data/books.csv"
books = spark.read.option("header", "true").csv("data/books.csv")
books.show(5, False)

+------------------------------------------------------------------+------------------------+-----------+-------+-----+----+-----------+
|Name                                                              |Author                  |User Rating|Reviews|Price|Year|Genre      |
+------------------------------------------------------------------+------------------------+-----------+-------+-----+----+-----------+
|10-Day Green Smoothie Cleanse                                     |JJ Smith                |4.7        |17350  |8    |2016|Non Fiction|
|11/22/63: A Novel                                                 |Stephen King            |4.6        |2052   |22   |2011|Fiction    |
|12 Rules for Life: An Antidote to Chaos                           |Jordan B. Peterson      |4.7        |18979  |15   |2018|Non Fiction|
|1984 (Signet Classics)                                            |George Orwell           |4.7        |21424  |6    |2017|Fiction    |
|5,000 Awesome Facts (About Everything!) 

In [11]:
books.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Author: string (nullable = true)
 |-- User Rating: string (nullable = true)
 |-- Reviews: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Genre: string (nullable = true)



In [12]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

booksSchema = StructType([
    StructField("Name", StringType(), nullable = True),
    StructField("Author", StringType(), nullable = True),
    StructField("User Rating", DoubleType(), nullable = True),
    StructField("Reviews", IntegerType(), nullable = True),
    StructField("Price", IntegerType(), nullable = True),
    StructField("Year", IntegerType(), nullable = True),
    StructField("Genre", StringType(), nullable = True)
])

In [13]:
booksSchemaDF = spark.read.option("header", "true").schema(booksSchema).csv("data/books.csv")
booksSchemaDF.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Author: string (nullable = true)
 |-- User Rating: double (nullable = true)
 |-- Reviews: integer (nullable = true)
 |-- Price: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Genre: string (nullable = true)



In [14]:
booksSchemaDF.show(5, False)

+------------------------------------------------------------------+------------------------+-----------+-------+-----+----+-----------+
|Name                                                              |Author                  |User Rating|Reviews|Price|Year|Genre      |
+------------------------------------------------------------------+------------------------+-----------+-------+-----+----+-----------+
|10-Day Green Smoothie Cleanse                                     |JJ Smith                |4.7        |17350  |8    |2016|Non Fiction|
|11/22/63: A Novel                                                 |Stephen King            |4.6        |2052   |22   |2011|Fiction    |
|12 Rules for Life: An Antidote to Chaos                           |Jordan B. Peterson      |4.7        |18979  |15   |2018|Non Fiction|
|1984 (Signet Classics)                                            |George Orwell           |4.7        |21424  |6    |2017|Fiction    |
|5,000 Awesome Facts (About Everything!) 

In [15]:
booksInferDF = spark.read.option("header", "true").option("inferSchema ", "true").csv("data/books.csv")
booksInferDF.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Author: string (nullable = true)
 |-- User Rating: string (nullable = true)
 |-- Reviews: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Genre: string (nullable = true)



## JSON

### Пример 1

In [16]:
!head -5 data/customer.json

{"CustomerID":"12346","Country":"United Kingdom","Address":"Unit 1047 Box 4089\nDPO AA 57348","Birthdate":"1994-02-20 00:46:27","Email":"cooperalexis@hotmail.com","Name":"Lindsay Cowan","Username":"valenciajennifer"}
{"CustomerID":"12347","Country":"Iceland","Address":"55711 Janet Plaza Apt. 865\nChristinachester, CT 62716","Birthdate":"1988-06-21 00:15:34","Email":"timothy78@hotmail.com","Name":"Katherine David","Username":"hillrachel"}
{"CustomerID":"12348","Country":"Finland","Address":"Unit 2676 Box 9352\nDPO AA 38560","Birthdate":"1974-11-26 15:30:20","Email":"tcrawford@gmail.com","Name":"Leslie Martinez","Username":"serranobrian"}
{"CustomerID":"12349","Country":"Italy","Address":"2765 Powers Meadow\nHeatherfurt, CT 53165","Birthdate":"1977-05-06 23:57:35","Email":"dustin37@yahoo.com","Name":"Brad Cardenas","Username":"charleshudson"}
{"CustomerID":"12350","Country":"Norway","Address":"17677 Mark Crest\nWalterberg, IA 39017","Birthdate":"1996-09-13 19:14:27","Email":"amyholland@y

In [17]:
!head -1 data/customer.json | jq

{
  "CustomerID": "12346",
  "Country": "United Kingdom",
  "Address": "Unit 1047 Box 4089\nDPO AA 57348",
  "Birthdate": "1994-02-20 00:46:27",
  "Email": "cooperalexis@hotmail.com",
  "Name": "Lindsay Cowan",
  "Username": "valenciajennifer"
}


In [18]:
customer = spark.read.json("data/customer.json")
customer.printSchema()

root
 |-- Address: string (nullable = true)
 |-- Birthdate: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Username: string (nullable = true)



In [19]:
customer.show(5, False)

+------------------------------------------------------+-------------------+--------------+----------+------------------------+---------------+----------------+
|Address                                               |Birthdate          |Country       |CustomerID|Email                   |Name           |Username        |
+------------------------------------------------------+-------------------+--------------+----------+------------------------+---------------+----------------+
|Unit 1047 Box 4089\nDPO AA 57348                      |1994-02-20 00:46:27|United Kingdom|12346     |cooperalexis@hotmail.com|Lindsay Cowan  |valenciajennifer|
|55711 Janet Plaza Apt. 865\nChristinachester, CT 62716|1988-06-21 00:15:34|Iceland       |12347     |timothy78@hotmail.com   |Katherine David|hillrachel      |
|Unit 2676 Box 9352\nDPO AA 38560                      |1974-11-26 15:30:20|Finland       |12348     |tcrawford@gmail.com     |Leslie Martinez|serranobrian    |
|2765 Powers Meadow\nHeatherfurt, 

### Пример 2

In [20]:
!head -20 data/countries.json

[
    {
        "name": {
            "common": "Aruba",
            "official": "Aruba",
            "native": {
                "nld": {
                    "official": "Aruba",
                    "common": "Aruba"
                },
                "pap": {
                    "official": "Aruba",
                    "common": "Aruba"
                }
            }
        },
        "tld": [
            ".aw"
        ],
        "cca2": "AW",


In [21]:
countries = spark\
    .read\
    .format("json")\
    .option("multiLine", "true")\
    .load("data/countries.json")

In [22]:
countries.show(5, False)

[Stage 15:>                                                         (0 + 1) / 1]

+-----------------------------------------------+---------+------------------------------+------------+----+----+----+----+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [23]:
countries.printSchema()

root
 |-- altSpellings: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- area: double (nullable = true)
 |-- borders: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- capital: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- cca2: string (nullable = true)
 |-- cca3: string (nullable = true)
 |-- ccn3: string (nullable = true)
 |-- cioc: string (nullable = true)
 |-- currencies: struct (nullable = true)
 |    |-- AED: struct (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- symbol: string (nullable = true)
 |    |-- AFN: struct (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- symbol: string (nullable = true)
 |    |-- ALL: struct (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- symbol: string (nullable = true)
 |    |-- AMD: struct (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- symbol:

## Parquet

In [24]:
!parquet-tools

usage: parquet-tools [-h] {show,csv,inspect} ...

parquet CLI tools

positional arguments:
  {show,csv,inspect}
    show              Show human readable format. see `show -h`
    csv               Cat csv style. see `csv -h`
    inspect           Inspect parquet file. see `inspect -h`

options:
  -h, --help          show this help message and exit


In [25]:
data1 = list(map(lambda x: [x, x * x], [1, 2, 3, 4, 5]))
squaresDF = spark.createDataFrame(data1).withColumnsRenamed({"_1": "value", "_2": "square"})
squaresDF.show()

+-----+------+
|value|square|
+-----+------+
|    1|     1|
|    2|     4|
|    3|     9|
|    4|    16|
|    5|    25|
+-----+------+



In [26]:
squaresDF.write.mode("overwrite").parquet("data/test_table/key=1")

In [27]:
data2 = list(map(lambda x: [x, x * x * x], [6, 7, 8, 9, 10]))
cubesDF = spark.createDataFrame(data2).withColumnsRenamed({"_1": "value", "_2": "cube"})
cubesDF.show()

+-----+----+
|value|cube|
+-----+----+
|    6| 216|
|    7| 343|
|    8| 512|
|    9| 729|
|   10|1000|
+-----+----+



In [28]:
cubesDF.write.mode("overwrite").parquet("data/test_table/key=2")

In [29]:
mergedDF = spark.read.option("mergeSchema", "true").parquet("data/test_table")
mergedDF.printSchema()

root
 |-- value: long (nullable = true)
 |-- square: long (nullable = true)
 |-- cube: long (nullable = true)
 |-- key: integer (nullable = true)



In [30]:
mergedDF.show()

+-----+------+----+---+
|value|square|cube|key|
+-----+------+----+---+
|    2|     4|NULL|  1|
|    4|    16|NULL|  1|
|    3|     9|NULL|  1|
|    5|    25|NULL|  1|
|    1|     1|NULL|  1|
|    6|  NULL| 216|  2|
|    7|  NULL| 343|  2|
|    9|  NULL| 729|  2|
|   10|  NULL|1000|  2|
|    8|  NULL| 512|  2|
+-----+------+----+---+



In [31]:
mergedDF.write.mode("overwrite").parquet("data/merged.parquet")

In [32]:
unMergedDF = spark.read.option("mergeSchema", "false").parquet("data/test_table")
unMergedDF.printSchema()

root
 |-- value: long (nullable = true)
 |-- square: long (nullable = true)
 |-- key: integer (nullable = true)



In [33]:
unMergedDF.show()

+-----+------+---+
|value|square|key|
+-----+------+---+
|    2|     4|  1|
|    4|    16|  1|
|    3|     9|  1|
|    5|    25|  1|
|    1|     1|  1|
|    6|  NULL|  2|
|    7|  NULL|  2|
|    9|  NULL|  2|
|   10|  NULL|  2|
|    8|  NULL|  2|
+-----+------+---+



In [34]:
!parquet-tools show data/merged.parquet

+---------+----------+--------+-------+
|   value |   square |   cube |   key |
|---------+----------+--------+-------|
|       2 |        4 |    nan |     1 |
|       4 |       16 |    nan |     1 |
|       3 |        9 |    nan |     1 |
|       5 |       25 |    nan |     1 |
|       1 |        1 |    nan |     1 |
|       6 |      nan |    216 |     2 |
|       7 |      nan |    343 |     2 |
|       9 |      nan |    729 |     2 |
|      10 |      nan |   1000 |     2 |
|       8 |      nan |    512 |     2 |
+---------+----------+--------+-------+


In [35]:
!ls -1 data/merged.parquet | grep -v _SUCCESS

part-00000-c28544ce-7424-411c-b999-bb30f1031ac6-c000.snappy.parquet
part-00001-c28544ce-7424-411c-b999-bb30f1031ac6-c000.snappy.parquet
part-00002-c28544ce-7424-411c-b999-bb30f1031ac6-c000.snappy.parquet
part-00003-c28544ce-7424-411c-b999-bb30f1031ac6-c000.snappy.parquet
part-00004-c28544ce-7424-411c-b999-bb30f1031ac6-c000.snappy.parquet


In [36]:
merged = %system ls -1 data/merged.parquet
file1 = merged[0]

In [37]:
print(file1)

part-00000-c28544ce-7424-411c-b999-bb30f1031ac6-c000.snappy.parquet


In [38]:
!parquet-tools inspect data/merged.parquet/{file1}


############ file meta data ############
created_by: parquet-mr version 1.13.1 (build db4183109d5b734ec5930d870cdae161e408ddba)
num_columns: 4
num_rows: 2
num_row_groups: 1
format_version: 1.0
serialized_size: 861


############ Columns ############
value
square
cube
key

############ Column(value) ############
name: value
path: value
max_definition_level: 1
max_repetition_level: 0
physical_type: INT64
logical_type: None
converted_type (legacy): NONE
compression: SNAPPY (space_saved: -2%)

############ Column(square) ############
name: square
path: square
max_definition_level: 1
max_repetition_level: 0
physical_type: INT64
logical_type: None
converted_type (legacy): NONE
compression: SNAPPY (space_saved: -4%)

############ Column(cube) ############
name: cube
path: cube
max_definition_level: 1
max_repetition_level: 0
physical_type: INT64
logical_type: None
converted_type (legacy): NONE
compression: SNAPPY (space_saved: -7%)

############ Column(key) ############
name: key
path: key
ma

In [39]:
!ls -1 data/test_table/key=2

part-00000-ef834c29-8b7f-413a-bd42-abaef30f5072-c000.snappy.parquet
part-00001-ef834c29-8b7f-413a-bd42-abaef30f5072-c000.snappy.parquet
part-00003-ef834c29-8b7f-413a-bd42-abaef30f5072-c000.snappy.parquet
part-00004-ef834c29-8b7f-413a-bd42-abaef30f5072-c000.snappy.parquet
part-00006-ef834c29-8b7f-413a-bd42-abaef30f5072-c000.snappy.parquet
part-00007-ef834c29-8b7f-413a-bd42-abaef30f5072-c000.snappy.parquet
_SUCCESS


In [40]:
key2Files = %system ls -1 data/test_table/key=2
file2 = key2Files[1]

In [41]:
!parquet-tools inspect data/test_table/key=2/{file2}


############ file meta data ############
created_by: parquet-mr version 1.13.1 (build db4183109d5b734ec5930d870cdae161e408ddba)
num_columns: 2
num_rows: 1
num_row_groups: 1
format_version: 1.0
serialized_size: 565


############ Columns ############
value
cube

############ Column(value) ############
name: value
path: value
max_definition_level: 1
max_repetition_level: 0
physical_type: INT64
logical_type: None
converted_type (legacy): NONE
compression: SNAPPY (space_saved: -5%)

############ Column(cube) ############
name: cube
path: cube
max_definition_level: 1
max_repetition_level: 0
physical_type: INT64
logical_type: None
converted_type (legacy): NONE
compression: SNAPPY (space_saved: -5%)



## ORC

In [42]:
!java17 -jar /opt/orc/orc-tools-2.3.0-uber.jar

ORC Java Tools

usage: java -jar orc-tools-*.jar [--help] [--define X=Y] <command> <args>

Commands:
   check - check the index of the specified column
   convert - convert CSV/JSON/ORC files to ORC
   count - recursively find *.orc and print the number of rows
   data - print the data from the ORC file
   json-schema - scan JSON files to determine their schema
   key - print information about the keys
   merge - merge multiple ORC files into a single ORC file
   meta - print the metadata about the ORC file
   scan - scan the ORC file
   sizes - list size on disk of each column
   version - print the version of this ORC tool

To get more help, provide -h to the command


### Пример 1

In [43]:
!java17 -jar /opt/orc/orc-tools-2.3.0-uber.jar data data/users.orc

Processing data file data/users.orc [length: 547]
{"name":"Alyssa","favorite_color":null,"favorite_numbers":[3,9,15,20]}
{"name":"Ben","favorite_color":"red","favorite_numbers":[]}
________________________________________________________________________________________________________________________



In [44]:
!java17 -jar /opt/orc/orc-tools-2.3.0-uber.jar meta data/users.orc

Processing data file data/users.orc [length: 547]
Structure for data/users.orc
File Version: 0.12 with ORC_135 by ORC Java 
Rows: 2
Compression: SNAPPY
Compression size: 262144
Calendar: Julian/Gregorian
Type: struct<name:string,favorite_color:string,favorite_numbers:array<int>>

Stripe Statistics:
  Stripe 1:
    Column 0: count: 2 hasNull: false
    Column 1: count: 2 hasNull: false bytesOnDisk: 18 min: Alyssa max: Ben sum: 9
    Column 2: count: 1 hasNull: true bytesOnDisk: 17 min: red max: red sum: 3
    Column 3: count: 2 hasNull: false bytesOnDisk: 6
    Column 4: count: 4 hasNull: false bytesOnDisk: 8 min: 3 max: 20 sum: 47

File Statistics:
  Column 0: count: 2 hasNull: false
  Column 1: count: 2 hasNull: false bytesOnDisk: 18 min: Alyssa max: Ben sum: 9
  Column 2: count: 1 hasNull: true bytesOnDisk: 17 min: red max: red sum: 3
  Column 3: count: 2 hasNull: false bytesOnDisk: 6
  Column 4: count: 4 hasNull: false bytesOnDisk: 8 min: 3 max: 20 sum: 47

Stripes:
  Stripe: offset

In [45]:
users = spark.read.orc("data/users.orc")
users.printSchema()

root
 |-- name: string (nullable = true)
 |-- favorite_color: string (nullable = true)
 |-- favorite_numbers: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [46]:
users.show()

+------+--------------+----------------+
|  name|favorite_color|favorite_numbers|
+------+--------------+----------------+
|Alyssa|          NULL|  [3, 9, 15, 20]|
|   Ben|           red|              []|
+------+--------------+----------------+



### Пример 2

In [47]:
!java17 -jar /opt/orc/orc-tools-2.3.0-uber.jar data data/example.orc

Processing data file data/example.orc [length: 490]
{"one":-1.0,"two":"foo","three":true}
{"one":NaN,"two":"bar","three":false}
{"one":2.5,"two":"baz","three":true}
________________________________________________________________________________________________________________________



In [48]:
!java17 -jar /opt/orc/orc-tools-2.3.0-uber.jar meta data/example.orc

Processing data file data/example.orc [length: 490]
Structure for data/example.orc
File Version: 0.12 with ORC_CPP_ORIGINAL by ORC C++ 2.1.0
Rows: 3
Compression: NONE
Calendar: Julian/Gregorian
Type: struct<one:double,two:string,three:boolean>

Stripe Statistics:
  Stripe 1:
    Column 0: count: 3 hasNull: false
    Column 1: count: 3 hasNull: false min: -1.0 max: 2.5 sum: NaN
    Column 2: count: 3 hasNull: false min: bar max: foo sum: 9
    Column 3: count: 3 hasNull: false true: 2

File Statistics:
  Column 0: count: 3 hasNull: false
  Column 1: count: 3 hasNull: false min: -1.0 max: 2.5 sum: NaN
  Column 2: count: 3 hasNull: false min: bar max: foo sum: 9
  Column 3: count: 3 hasNull: false true: 2

Stripes:
  Stripe: offset: 3 data: 37 rows: 3 tail: 93 index: 93
    Stream: column 0 section ROW_INDEX start: 3 length 8
    Stream: column 1 section ROW_INDEX start: 11 length 40
    Stream: column 2 section ROW_INDEX start: 51 length 27
    Stream: column 3 section ROW_INDEX start: 7

In [49]:
example = spark.read.orc("data/example.orc")
example.printSchema()

root
 |-- one: double (nullable = true)
 |-- two: string (nullable = true)
 |-- three: boolean (nullable = true)



In [50]:
example.show()

+----+---+-----+
| one|two|three|
+----+---+-----+
|-1.0|foo| true|
| NaN|bar|false|
| 2.5|baz| true|
+----+---+-----+



## Avro

In [51]:
!java -jar /opt/avro/avro-tools-1.12.1.jar

Version 1.12.1
 of Apache Avro
Copyright 2010-2015 The Apache Software Foundation

This product includes software developed at
The Apache Software Foundation (https://www.apache.org/).
----------------
Available tools:
    canonical  Converts an Avro Schema to its canonical form
          cat  Extracts samples from files
      compile  Generates Java code for the given schema.
       concat  Concatenates avro files without re-compressing.
        count  Counts the records in avro files or folders
  fingerprint  Returns the fingerprint for the schemas.
   fragtojson  Renders a binary-encoded Avro datum as JSON.
     fromjson  Reads JSON records and writes an Avro data file.
     fromtext  Imports a text file into an avro data file.
      getmeta  Prints out the metadata of an Avro data file.
    getschema  Prints out schema of an Avro data file.
          idl  Generates a JSON schema or protocol from an Avro IDL file
 idl2schemata  Extract JSON schemata of the types from an Avro IDL fil

In [52]:
!java -jar /opt/avro/avro-tools-1.12.1.jar getschema data/users.avro

26/06/28 16:04:24 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
{
  "type" : "record",
  "name" : "User",
  "namespace" : "example.avro",
  "fields" : [ {
    "name" : "name",
    "type" : "string"
  }, {
    "name" : "favorite_color",
    "type" : [ "string", "null" ]
  }, {
    "name" : "favorite_numbers",
    "type" : {
      "type" : "array",
      "items" : "int"
    }
  } ]
}


In [53]:
!java -jar /opt/avro/avro-tools-1.12.1.jar tojson data/users.avro

26/06/28 16:04:24 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
{"name":"Alyssa","favorite_color":null,"favorite_numbers":[3,9,15,20]}
{"name":"Ben","favorite_color":{"string":"red"},"favorite_numbers":[]}


In [54]:
usersDF = spark.read.format("avro").load("data/users.avro")
usersDF.printSchema()

root
 |-- name: string (nullable = true)
 |-- favorite_color: string (nullable = true)
 |-- favorite_numbers: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [55]:
usersDF.show()

+------+--------------+----------------+
|  name|favorite_color|favorite_numbers|
+------+--------------+----------------+
|Alyssa|          NULL|  [3, 9, 15, 20]|
|   Ben|           red|              []|
+------+--------------+----------------+



In [56]:
usersDF\
    .select("name", "favorite_color")\
    .write\
    .mode("overwrite")\
    .format("avro")\
    .save("data/namesAndFavColors.avro")

In [57]:
avroFiles = %system ls -1 data/namesAndFavColors.avro | grep -v _SUCCESS
avroFile = avroFiles[0]

!java -jar /opt/avro/avro-tools-1.12.1.jar getmeta data/namesAndFavColors.avro/{avroFile}

26/06/28 16:04:26 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
avro.schema	{"type":"record","name":"topLevelRecord","fields":[{"name":"name","type":["string","null"]},{"name":"favorite_color","type":["string","null"]}]}
org.apache.spark.version	3.5.8
avro.codec	snappy


In [58]:
namesAndFavColorsDF = spark.read.format("avro").load("data/namesAndFavColors.avro")
namesAndFavColorsDF.printSchema()

root
 |-- name: string (nullable = true)
 |-- favorite_color: string (nullable = true)



In [59]:
namesAndFavColorsDF.show()

+------+--------------+
|  name|favorite_color|
+------+--------------+
|Alyssa|          NULL|
|   Ben|           red|
+------+--------------+



In [60]:
!java -jar /opt/avro/avro-tools-1.12.1.jar compile schema data/user.avsc data

Input files to compile:
  data/user.avsc


In [61]:
!head data/example/avro/User.java

/*
 * Autogenerated by Avro
 *
 * DO NOT EDIT DIRECTLY
 */
package example.avro;

import org.apache.avro.specific.SpecificData;
import org.apache.avro.util.Utf8;
import org.apache.avro.message.BinaryMessageEncoder;
